In [1]:
import pandas as pd
df = pd.read_csv('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/GNN-SOM-master/123_site_from_smiles.csv')
substrates = df['substrate'].to_list()
ecs = [(1,14) for _ in range(len(substrates))]
# dict_sub2ec = dict(zip(substrates,ecs))
# ecs = [i.strip() for i in ecs]

# # 凡是没有EC的我们用1.14代替
# ecs = [i.replace('None','1.14') for i in ecs]
# ecs = [i.split(' ') for i in ecs]
# ecs_new = []
# for eclist in ecs:
#     eclist = [tuple([int(j) for j in i.split('.')[:2]]) for i in eclist]
#     eclist = list(set(eclist))
#     ecs_new.append(eclist)

# subs_new = []
# for i in range(len(substrates)):
#     ecs = ecs_new[i]
#     num_repeat = len(ecs)
#     subs = [substrates[i] for _ in range(num_repeat)]
#     subs_new = subs_new + subs

# ecs_new = [i for k in ecs_new for i in k]

df_new = pd.DataFrame(columns=['substrate','ec','score'])
df_new['substrate'] = substrates
df_new['ec'] = ecs
df_new['score'] = ['None' for _ in range(len(substrates))]
df_new


,substrate,ec,score
0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,"(1, 14)",None
1,CC(=O)NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2,"(1, 14)",None
2,CC(=O)NC1=CC=CC(N2C(=O)N(C3CC3)C(=O)C3=C(NC4=C...,"(1, 14)",None
3,CC(C)(C)C1=CC(C(C)(C)C)=C(NC(=O)C2=CNC3=CC=CC=...,"(1, 14)",None
4,CC(C)(C)C1=CC(NC(=O)NC2=CC=C(C3=CN4C(=N3)SC3=C...,"(1, 14)",None
...,...,...,...
118,OC(C1=CC=CC=C1)(C1=CC=CC=C1)C12CC[N+](CCOCC3=C...,"(1, 14)",None
119,OC[C@H]1O[C@@H](C2=CC=C(Cl)C(CC3=CC=C(OCCOC4CC...,"(1, 14)",None
120,OCC1=NC=C2N1C1=CC=C(Cl)C=C1C(C1=CC=CC=C1F)=NC2,"(1, 14)",None
121,OC1=CC=C(COCC[N+]23CCC(C(O)(C4=CC=CC=C4)C4=CC=...,"(1, 14)",None


In [4]:
from tqdm import tqdm

import json
from os.path import isfile

import torch
from torch_geometric.data import Data
from rdkit.Chem import Draw
import requests

from IPython.display import SVG

from gnn_som import createGnnSom, loadGnnSomState
from gnn_som.MolFromKcf import MolFromKcfFile

import os

from rdkit import Chem
from rdkit.Chem import AllChem

os.environ['CUDA_VISIBLE_DEVICES'] = "2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open('data/config.json', 'r') as f:
    config = json.load(f)
config['features']['enzyme'] = [tuple(ec) for ec in config['features']['enzyme']] 

models = []
for i, params in enumerate(config['models']):
    model = createGnnSom(*config['models'][i])
    loadGnnSomState(model, torch.load('data/model%d.pt' % i, map_location=torch.device('cpu')))
    models.append(model)

# Convert SMILES to .mol file using RDKit 

atom_scores_list = []

for i in tqdm(range(len(df_new))):
    smi = df_new.iloc[i,0]
    enzyme = df_new.iloc[i,1]


    # Create RDKit mol object
    # molecule = 'CC1([C@@H]2[C@H]1[C@H](N(C2)C(=O)[C@H](C(C)(C)C)NC(=O)C(F)(F)F)C(=O)N[C@@H](C[C@@H]3CCNC3=O)C#N)C'
    print(smi)
    molecule = Chem.MolFromSmiles(smi)
    molecule = Chem.AddHs(molecule)

    if os.path.exists("molecule.mol"):
        os.remove("molecule.mol")

    # Compute 2D coordenates and write to .mol file
    AllChem.Compute2DCoords(molecule)
    print(Chem.MolToMolBlock(molecule), file=open("molecule.mol", 'a+'))

    if os.path.exists("molecule.kcf"):
        os.remove("molecule.kcf")

    # Convert .mol to .kcf (KEGG format used in the paper) 
    # https://iwatobipen.wordpress.com/2016/12/23/convert-chemical-file-format/
    # https://www.genome.jp/tools/gn_ca_tools_api.html
    !curl -F molfile=@molecule.mol http://rest.genome.jp/mol2kcf/ > molecule.kcf

    # Define mol
    mol = MolFromKcfFile('molecule.kcf')

        # enzyme = (2,4)

    print(enzyme)
    numFeatures = sum(len(feature) for feature in config['features'].values())
    x = torch.zeros((mol.GetNumAtoms(), numFeatures), dtype=torch.float32)
    for atom in mol.GetAtoms():
        x[atom.GetIdx(), config['features']['enzyme'].index(enzyme)] = 1
        offset = len(config['features']['enzyme'])
        x[atom.GetIdx(), offset + config['features']['element'].index(atom.GetSymbol())] = 1
        offset += len(config['features']['element'])
        x[atom.GetIdx(), offset + config['features']['kcfType'].index(atom.GetProp('kcfType'))] = 1

    edgeIndex = torch.zeros((2, mol.GetNumBonds() * 2), dtype=torch.int64)
    for bond in mol.GetBonds():
        i = bond.GetIdx()
        edgeIndex[0][i * 2] = bond.GetBeginAtomIdx()
        edgeIndex[1][i * 2] = bond.GetEndAtomIdx()
        edgeIndex[0][i * 2 + 1] = bond.GetEndAtomIdx()
        edgeIndex[1][i * 2 + 1] = bond.GetBeginAtomIdx()

    data = Data(x=x, edgeIndex=edgeIndex)

    # 确保模型和数据都在同一设备上运行
    data = data.to(device)  # 将数据移动到 GPU 或保持在 CPU
    y = None

    for model in models:
        model = model.to(device)  # 将每个模型移动到 GPU（如果尚未移动）
        newY = torch.sigmoid(model(data.x, data.edgeIndex))  # 数据已在 GPU 或 CPU 上
        y = newY if y is None else torch.add(y, newY)

    # 计算模型平均输出
    y = torch.div(y, len(models))
    print(y.shape)

    # import torch

    # # 假设 scores 是你的 torch 张量，大小是 [14, 1]
    # scores = y  # 示例：14个原子的反应分数，实际使用时替换为你的数据

    # # 创建一个空字典来存储原子序号和对应的分数
    # atom_scores = {}

    # # 将分数映射到原子序号
    # for idx, score in enumerate(scores):
    #     atom_scores[idx] = score.item()  # 使用 item() 将张量转换为标量

    # 打印结果
    # print(atom_scores)

    atom_scores_list.append(y)

    print(f'substrate: {smi}, ec: {enzyme}, score: {y}')

/tmp/ipykernel_352752/3440858778.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loadGnnSomState(model, torch.load('data/model%d.pt' % i, map_location=torch.device('cpu

C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@H]4[C@H]3CC[C@@]21CC
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  7167    0  2034  100  5133   4570  11534 --:--:-- --:--:-- --:--:-- 16105
(1, 14)


  1%|▏         | 1/72 [00:02<02:35,  2.18s/it]

torch.Size([27, 1])
substrate: C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@H]4[C@H]3CC[C@@]21CC, ec: (1, 14), score: tensor([[0.1723],
        [0.2866],
        [0.5596],
        [0.6462],
        [0.4943],
        [0.0226],
        [0.0584],
        [0.1977],
        [0.0186],
        [0.0059],
        [0.0049],
        [0.0308],
        [0.0327],
        [0.0077],
        [0.0733],
        [0.1889],
        [0.4892],
        [0.5515],
        [0.1425],
        [0.0872],
        [0.0181],
        [0.0025],
        [0.0077],
        [0.0172],
        [0.0121],
        [0.0360],
        [0.0486]], device='cuda:0', grad_fn=<DivBackward0>)
CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc(/C=C/c2ccc3ccc(Cl)cc3n2)c1
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  9739    0  3016  100  6723   5026  11205 --:--:-- --:--:-- --:--:-- 16231
(2, 4)


  3%|▎         | 2/72 [00:03<01:51,  1.59s/it]

...

torch.Size([20, 1])
substrate: O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O, ec: (2, 4), score: tensor([[9.9079e-02],
        [1.8771e-01],
        [2.4033e-01],
        [1.5612e-03],
        [1.1937e-04],
        [4.5815e-04],
        [2.7366e-03],
        [1.0821e-02],
        [2.6935e-04],
        [5.4540e-04],
        [3.2846e-02],
        [7.4971e-05],
        [4.0471e-04],
        [1.4493e-04],
        [7.5622e-04],
        [2.1055e-04],
        [7.2798e-05],
        [4.2656e-01],
        [3.9386e-01],
        [7.4765e-01]], device='cuda:0', grad_fn=<DivBackward0>)
O=P1(NCCCl)OCCCN1CCCl
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3736    0  1049  100  2687   3629   9297 --:--:-- --:--:-- --:--:-- 12882
(1, 14)


 99%|█████████▊| 71/72 [01:22<00:01,  1.12s/it]

torch.Size([14, 1])
substrate: O=P1(NCCCl)OCCCN1CCCl, ec: (1, 14), score: tensor([[0.1688],
        [0.0838],
        [0.9356],
        [0.8639],
        [0.0118],
        [0.0059],
        [0.0057],
        [0.0171],
        [0.0044],
        [0.4192],
        [0.8490],
        [0.7811],
        [0.1128],
        [0.0271]], device='cuda:0', grad_fn=<DivBackward0>)
OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6634    0  1929  100  4705   2028   4947 --:--:-- --:--:-- --:--:--  6968
(1, 14)


100%|██████████| 72/72 [01:24<00:00,  1.17s/it]

torch.Size([26, 1])
substrate: OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1, ec: (1, 14), score: tensor([[0.8060],
        [0.4816],
        [0.1687],
        [0.2425],
        [0.2999],
        [0.0622],
        [0.2464],
        [0.1973],
        [0.3632],
        [0.2738],
        [0.3328],
        [0.0274],
        [0.1121],
        [0.0789],
        [0.2265],
        [0.0971],
        [0.1053],
        [0.0570],
        [0.1969],
        [0.1583],
        [0.1666],
        [0.2654],
        [0.1717],
        [0.2384],
        [0.3145],
        [0.2815]], device='cuda:0', grad_fn=<DivBackward0>)


In [8]:
import torch
atom_scores_list_new = []
# 假设 scores 是你的 torch 张量，大小是 [14, 1]
for i in atom_scores_list:
    scores = i  # 示例：14个原子的反应分数，实际使用时替换为你的数据

    # 创建一个空字典来存储原子序号和对应的分数
    atom_scores = {}

    # 将分数映射到原子序号
    for idx, score in enumerate(scores):
        atom_scores[idx] = score.item()  # 使用 item() 将张量转换为标量

# 打印结果
    atom_scores_list_new.append(atom_scores)

In [11]:
df_new['score'] = atom_scores_list_new

,substrate,ec,score
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,"(1, 14)","{0: 0.17234759032726288, 1: 0.2866118252277374..."
1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"(2, 4)","{0: 0.03460502251982689, 1: 0.1962412446737289..."
2,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"(1, 14)","{0: 0.26034024357795715, 1: 0.9417978525161743..."
3,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,"(1, 14)","{0: 0.07399298995733261, 1: 0.0268323067575693..."
4,CC(CN1c2ccccc2Sc2ccccc21)N(C)C,"(1, 14)","{0: 0.06365552544593811, 1: 0.2708439826965332..."
...,...,...,...
67,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,"(1, 14)","{0: 0.015096035785973072, 1: 0.234839916229248..."
68,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,"(1, 14)","{0: 0.6005780696868896, 1: 0.9662299156188965,..."
69,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,"(2, 4)","{0: 0.09907922148704529, 1: 0.1877077072858810..."
70,O=P1(NCCCl)OCCCN1CCCl,"(1, 14)","{0: 0.16878382861614227, 1: 0.0837636962532997..."


In [15]:
df_new.to_pickle('gnnsom_results.pickle')